In [9]:
!pip install -q -U transformers accelerate bitsandbytes

In [10]:
!pip install -q chainlit pyngrok

In [11]:
!pip install chainlit

In [12]:
!pip install -q \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    pypdf

In [13]:
%%writefile app.py

import torch
import chainlit as cl

from pypdf import PdfReader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# =========================================================
# 1. LOAD QWEN
# =========================================================

MODEL_NAME = "Qwen/Qwen3-4B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    return_full_text=False,
)


# =========================================================
# 2. EMBEDDING MODEL
# =========================================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# =========================================================
# 3. HÀM ĐỌC PDF
# =========================================================

def load_pdf(file_path: str):
    reader = PdfReader(file_path)

    documents = []

    for page_number, page in enumerate(reader.pages):
        text = page.extract_text()

        if text and text.strip():
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": file_path,
                        "page": page_number + 1,
                    },
                )
            )

    return documents


# =========================================================
# 4. TẠO RETRIEVER
# =========================================================

def create_retriever(file_path: str):
    documents = load_pdf(file_path)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )

    chunks = text_splitter.split_documents(documents)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 4}
    )

    return retriever, len(documents), len(chunks)

# =========================================================
# 5. KHI USER MỞ CHAT
# =========================================================

@cl.on_chat_start
async def on_chat_start():

    files = None

    while files is None:
        files = await cl.AskFileMessage(
            content="Upload a PDF file to start chatting with it.",
            accept=["application/pdf"],
            max_size_mb=20,
            max_files=1,
        ).send()

    pdf_file = files[0]

    await cl.Message(
        content=f"Processing `{pdf_file.name}`..."
    ).send()

    retriever, num_pages, num_chunks = create_retriever(
        pdf_file.path
    )

    # Lưu retriever riêng cho session hiện tại
    cl.user_session.set("retriever", retriever)

    # Lưu lịch sử hội thoại
    cl.user_session.set("chat_history", [])

    await cl.Message(
        content=(
            f"`{pdf_file.name}` is ready.\n\n"
            f"- Pages: {num_pages}\n"
            f"- Chunks: {num_chunks}\n\n"
            "You can now ask questions about the document."
        )
    ).send()

# =========================================================
# 6. KHI USER GỬI MESSAGE
# =========================================================

@cl.on_message
async def on_message(message: cl.Message):

    question = message.content

    retriever = cl.user_session.get("retriever")
    chat_history = cl.user_session.get("chat_history")

    if retriever is None:
        await cl.Message(
            content="Please upload a PDF first."
        ).send()
        return

    # -------------------------
    # Retrieval
    # -------------------------

    relevant_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in relevant_docs
    )

    # -------------------------
    # Prompt cho lượt hiện tại
    # -------------------------

    inference_messages = chat_history.copy()

    inference_messages.append(
        {
            "role": "user",
            "content": f"""
You are a question-answering assistant.

Answer the question using the provided document context.

If the answer cannot be found in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}
""",
        }
    )

    prompt = tokenizer.apply_chat_template(
        inference_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # -------------------------
    # Qwen generate
    # -------------------------

    output = model_pipeline(prompt)

    answer = output[0]["generated_text"].strip()

    # -------------------------
    # Lưu chat history
    # -------------------------

    chat_history.append(
        {
            "role": "user",
            "content": question,
        }
    )

    chat_history.append(
        {
            "role": "assistant",
            "content": answer,
        }
    )

    cl.user_session.set(
        "chat_history",
        chat_history,
    )

    # -------------------------
    # Hiện câu trả lời
    # -------------------------

    await cl.Message(
        content=answer
    ).send()

Overwriting app.py


In [14]:
from pyngrok import ngrok
NGROK_TOKEN = "3HJHWk0Z2dkEQ6ohl3vUarMtl5k_HWqB63uffuRBXeqb7YKV"

ngrok.set_auth_token(NGROK_TOKEN)

In [15]:
!chainlit run app.py --port 8000 > /tmp/chainlit.log 2>&1 &

In [16]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://unrented-unroll-antitoxic.ngrok-free.dev" -> "http://localhost:8000"
